# Imaging Evaluation — Candidate Comparison, Promotion, Confound Check

In [1]:
import sys
sys.path.insert(0, r"C:\FYP\src")
from utils.config import (
    RESULTS_IMAGING_DIR, IMAGING_MODEL_COMPARISON_PATH, IMAGING_OOF_PREDICTIONS_PATH,
    CHECKPOINTS_IMAGING_CANDIDATES_DIR, CHECKPOINTS_IMAGING_FINAL_DIR, EVAL_IMAGING_DIR,
    CACHE_IMAGES_PATH, CACHE_MASKS_PATH, CACHE_MANIFEST_PATH, CACHE_META_PATH,
    ensure_dirs,
)
from imaging.models import ResNet50UNet
from imaging.slice_cache_dataset import SliceCacheDataset

import json
from datetime import date

import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score

ensure_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Imports OK, device: {DEVICE}")

Imports OK, device: cuda


## Load Per-Fold Metrics, Compare Candidates

In [2]:
CANDIDATES = ["resnet50_unet", "baseline_cnn"]


def load_model_card(candidate: str) -> dict:
    with open(CHECKPOINTS_IMAGING_CANDIDATES_DIR / candidate / "model_card.json") as f:
        return json.load(f)


def aggregate_candidate_metrics(candidate: str, card: dict) -> dict:
    """Mean/std across folds actually run, for every detection metric plus dice/iou
    (None-safe -- baseline_cnn has no segmentation head) and a pooled confusion matrix."""
    folds = card["per_fold"]
    row = {"model": candidate, "n_folds": len(folds)}

    for m in ["precision", "recall", "specificity", "f1", "roc_auc"]:
        vals = np.array([f["detection_metrics"][m] for f in folds], dtype=float)
        row[f"{m}_mean"] = float(np.nanmean(vals))
        row[f"{m}_std"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan

    dice_vals = [f["segmentation_metrics"]["dice"] for f in folds if f["segmentation_metrics"] is not None]
    iou_vals = [f["segmentation_metrics"]["iou"] for f in folds if f["segmentation_metrics"] is not None]
    row["dice_mean"] = float(np.mean(dice_vals)) if dice_vals else np.nan
    row["dice_std"] = float(np.std(dice_vals, ddof=1)) if len(dice_vals) > 1 else np.nan
    row["iou_mean"] = float(np.mean(iou_vals)) if iou_vals else np.nan
    row["iou_std"] = float(np.std(iou_vals, ddof=1)) if len(iou_vals) > 1 else np.nan

    cm_sum = np.sum([np.array(f["detection_metrics"]["confusion_matrix"]) for f in folds], axis=0)
    tn, fp, fn, tp = cm_sum.ravel()
    row["tn"], row["fp"], row["fn"], row["tp"] = int(tn), int(fp), int(fn), int(tp)
    return row


cards = {c: load_model_card(c) for c in CANDIDATES}
model_comparison_df = pd.DataFrame([aggregate_candidate_metrics(c, cards[c]) for c in CANDIDATES])
model_comparison_df.insert(0, "scheme", "5-Fold CV")
model_comparison_df.to_csv(IMAGING_MODEL_COMPARISON_PATH, index=False)

print(f"Wrote {IMAGING_MODEL_COMPARISON_PATH}")
print(model_comparison_df.to_string(index=False))

Wrote C:\FYP\results\imaging\model_comparison.csv
   scheme         model  n_folds  precision_mean  precision_std  recall_mean  recall_std  specificity_mean  specificity_std  f1_mean   f1_std  roc_auc_mean  roc_auc_std  dice_mean  dice_std  iou_mean  iou_std    tn   fp   fn    tp
5-Fold CV resnet50_unet        5        0.991981       0.009091     0.962841    0.037153          0.969637         0.033266 0.976785 0.016409      0.993425     0.006879   0.396432  0.204396   0.28349 0.152912 18031  585 2679 69398
5-Fold CV  baseline_cnn        5        0.964674       0.045346     0.907307    0.078942          0.858407         0.191323 0.932045 0.029270      0.984024     0.007611        NaN       NaN       NaN      NaN 15982 2634 6829 65248


In [3]:
oof_df = pd.read_csv(IMAGING_OOF_PREDICTIONS_PATH)
print(f"oof_predictions: {len(oof_df):,} rows")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for candidate in CANDIDATES:
    sub = oof_df[oof_df["candidate"] == candidate]
    y_true, y_proba = sub["y_true"].to_numpy(), sub["y_proba"].to_numpy()

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f"{candidate} (AUC={roc_auc_score(y_true, y_proba):.4f})")

    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    axes[1].plot(rec, prec, label=candidate)

    bin_edges = np.linspace(0, 1, 11)
    bin_ids = np.clip(np.digitize(y_proba, bin_edges) - 1, 0, 9)
    bin_obs = [y_true[bin_ids == i].mean() if (bin_ids == i).any() else np.nan for i in range(10)]
    bin_pred = [y_proba[bin_ids == i].mean() if (bin_ids == i).any() else np.nan for i in range(10)]
    axes[2].plot(bin_pred, bin_obs, marker="o", label=candidate)

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC (pooled OOF, all folds)"); axes[0].legend()

axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall (pooled OOF)"); axes[1].legend()

axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("Mean predicted probability"); axes[2].set_ylabel("Observed frequency")
axes[2].set_title("Reliability diagram (UNCALIBRATED, pooled OOF)"); axes[2].legend()

fig.tight_layout()
fig_path = EVAL_IMAGING_DIR / "candidate_comparison.png"
fig.savefig(fig_path, dpi=100)
plt.close(fig)
print(f"Saved {fig_path}")

oof_predictions: 181,386 rows

Saved C:\FYP\outputs\eval\imaging\candidate_comparison.png


## Confusion Matrix -- Both Candidates (complements the ROC/PR/reliability panel above)

In [ ]:
from sklearn.metrics import confusion_matrix

cm_fig, cm_axes = plt.subplots(1, 2, figsize=(12, 5.5))

for ax, candidate in zip(cm_axes, CANDIDATES):
    sub = oof_df[oof_df["candidate"] == candidate]
    cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

    ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm_pct[i, j]:.1f}%\n(n={cm[i, j]:,})", ha="center", va="center",
                    color="white" if cm_pct[i, j] > 55 else "black", fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No tumour", "Tumour"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No tumour", "Tumour"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{candidate}\n(threshold=0.5, pooled OOF, all folds)")

cm_fig.suptitle("Imaging Candidate Comparison -- Confusion Matrix (slice-level, pooled 5-fold OOF)", fontsize=13)
cm_fig.tight_layout(rect=[0, 0, 1, 0.94])
cm_fig_path = EVAL_IMAGING_DIR / "candidate_confusion_matrices.png"
cm_fig.savefig(cm_fig_path, dpi=100)
plt.close(cm_fig)
print(f"Saved {cm_fig_path}")

## The Winning Model: `resnet50_unet`

In [4]:
WINNER_MODEL_NAME = "resnet50_unet"

print(f"Winning model: {WINNER_MODEL_NAME}")
print("\nSupporting numbers (recall = sensitivity, the metric that decided this):")
print(model_comparison_df.set_index("model")[["recall_mean", "roc_auc_mean", "dice_mean"]].to_string())
print(
    "\nReasoning: resnet50_unet has higher, far more fold-stable recall than baseline_cnn "
    "(see per-fold breakdown in docs/Imaging_5Fold_Training_Results_documentation.md), and is "
    "the only candidate producing segmentation output the project's derived risk-score "
    "(tumour/gland pixel ratio) structurally depends on. baseline_cnn's recall/specificity "
    "swings dramatically fold-to-fold -- a liability, not just a tradeoff, for a model meant "
    "to generalize to unseen patients."
)

Winning model: resnet50_unet

Supporting numbers (recall = sensitivity, the metric that decided this):
               recall_mean  roc_auc_mean  dice_mean
model                                              
resnet50_unet     0.962841      0.993425   0.396432
baseline_cnn      0.907307      0.984024        NaN

Reasoning: resnet50_unet has higher, far more fold-stable recall than baseline_cnn (see per-fold breakdown in docs/Imaging_5Fold_Training_Results_documentation.md), and is the only candidate producing segmentation output the project's derived risk-score (tumour/gland pixel ratio) structurally depends on. baseline_cnn's recall/specificity swings dramatically fold-to-fold -- a liability, not just a tradeoff, for a model meant to generalize to unseen patients.


## Promote Winner to `checkpoints/imaging/final/`

In [5]:
# Step 1: verify the promoted checkpoint loads cleanly
model_pt_path = CHECKPOINTS_IMAGING_FINAL_DIR / "model.pt"
assert model_pt_path.exists(), f"{model_pt_path} not found -- run the pod final-fit step first"

state_dict = torch.load(model_pt_path, map_location=DEVICE, weights_only=True)
bad_keys = [k for k in state_dict.keys() if k.startswith("_orig_mod.")]
assert not bad_keys, f"torch.compile prefix leakage detected: {bad_keys[:5]}"

final_model = ResNet50UNet().to(DEVICE)
final_model.load_state_dict(state_dict)
final_model.eval()
n_params = sum(p.numel() for p in final_model.parameters())
print(f"Loaded {model_pt_path} into ResNet50UNet -- {n_params:,} params, no _orig_mod. leakage.")

with open(CHECKPOINTS_IMAGING_FINAL_DIR / "pod_training_run.json") as f:
    pod_run = json.load(f)
print("\npod_training_run.json (training provenance):")
print(json.dumps({k: v for k, v in pod_run.items() if k not in ("history_full", "per_epoch_train_losses")}, indent=2))

Loaded C:\FYP\checkpoints\imaging\final\model.pt into ResNet50UNet -- 71,876,484 params, no _orig_mod. leakage.

pod_training_run.json (training provenance):
{
  "candidate": "resnet50_unet",
  "fit_type": "final_all_data_no_folds",
  "n_patients": 361,
  "n_rows": 90693,
  "n_epochs": 6,
  "epoch_count_reasoning": "no held-out validation set exists for early stopping in a no-folds fit; used a fixed epoch count instead. Source: checkpoints/imaging/candidates/resnet50_unet/model_card.json per-fold best_epoch values [4, 0, 2, 12, 8], mean=5.2 -> round(5.2)=5 (0-indexed) -> 6 total epochs (epoch indices 0..5).",
  "batch_size": 32,
  "augment": true,
  "slice_stride": 3,
  "learning_rate": 0.0001,
  "optimizer": "AdamW",
  "use_torch_compile": true,
  "box_size": 320,
  "n_params": 71876484,
  "note": "per-epoch losses are a sanity check that training progressed normally, NOT a performance estimate -- there is no held-out data in this run to evaluate against. Calibration and model_card.js

In [6]:
# Step 2: calibrator -- Platt scaling, fit on the winner's OOF predictions only
winner_oof = oof_df[oof_df["candidate"] == WINNER_MODEL_NAME]
assert len(winner_oof) > 0, f"No OOF rows found for {WINNER_MODEL_NAME}"

X_calib = winner_oof[["y_proba"]].to_numpy()
y_calib = winner_oof["y_true"].to_numpy()

calibrator = LogisticRegression()
calibrator.fit(X_calib, y_calib)
joblib.dump(calibrator, CHECKPOINTS_IMAGING_FINAL_DIR / "calibrator.pkl")

print(
    f"Calibrator (Platt scaling) fit on {len(winner_oof):,} {WINNER_MODEL_NAME} OOF rows "
    f"({winner_oof['patient_id'].nunique()} unique patients)."
)

Calibrator (Platt scaling) fit on 90,693 resnet50_unet OOF rows (361 unique patients).


## Patient-Level (Whole-Volume) ROC, Precision-Recall, and Confusion Matrix -- `resnet50_unet` Only

In [ ]:
import textwrap

winner_oof = winner_oof.copy()
winner_oof["p"] = calibrator.predict_proba(winner_oof[["y_proba"]].to_numpy())[:, 1]

assert (winner_oof.groupby("patient_id")["fold"].nunique() > 1).sum() == 0
assert (winner_oof.groupby("patient_id")["y_true"].nunique() > 1).sum() == 0

patient_score = winner_oof.groupby("patient_id")["p"].mean()
patient_label = winner_oof.groupby("patient_id")["y_true"].first().loc[patient_score.index]

pl_y_true, pl_y_score = patient_label.to_numpy(), patient_score.to_numpy()
pl_y_pred = (pl_y_score >= 0.5).astype(int)

pl_fig, pl_axes = plt.subplots(1, 3, figsize=(18, 5.2))

pl_fpr, pl_tpr, _ = roc_curve(pl_y_true, pl_y_score)
pl_axes[0].plot(pl_fpr, pl_tpr, color="C0", label=f"{WINNER_MODEL_NAME} (AUC={roc_auc_score(pl_y_true, pl_y_score):.4f})")
pl_axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
pl_axes[0].set_xlabel("False Positive Rate"); pl_axes[0].set_ylabel("True Positive Rate")
pl_axes[0].set_title(f"ROC -- patient-level (n={len(pl_y_true)})"); pl_axes[0].legend(loc="lower right")

pl_prec, pl_rec, _ = precision_recall_curve(pl_y_true, pl_y_score)
pl_axes[1].plot(pl_rec, pl_prec, color="C0")
pl_axes[1].set_xlabel("Recall"); pl_axes[1].set_ylabel("Precision")
pl_axes[1].set_title("Precision-Recall -- patient-level")

pl_cm = confusion_matrix(pl_y_true, pl_y_pred, labels=[0, 1])
pl_cm_pct = pl_cm / pl_cm.sum(axis=1, keepdims=True) * 100
pl_axes[2].imshow(pl_cm_pct, cmap="Blues", vmin=0, vmax=100)
for i in range(2):
    for j in range(2):
        pl_axes[2].text(j, i, f"{pl_cm_pct[i, j]:.1f}%\n(n={pl_cm[i, j]})", ha="center", va="center",
                         color="white" if pl_cm_pct[i, j] > 55 else "black", fontsize=10)
pl_axes[2].set_xticks([0, 1]); pl_axes[2].set_xticklabels(["No tumour", "Tumour"])
pl_axes[2].set_yticks([0, 1]); pl_axes[2].set_yticklabels(["No tumour", "Tumour"])
pl_axes[2].set_xlabel("Predicted"); pl_axes[2].set_ylabel("True")
pl_axes[2].set_title("Confusion Matrix -- patient-level (threshold=0.5)")

pl_fig.suptitle(
    f"{WINNER_MODEL_NAME}: patient-level (whole-volume, mean-aggregated, calibrated) evaluation\n"
    "NOT slice-level -- one point per patient, aggregating that patient's calibrated slice scores",
    fontsize=12,
)
pl_fig.text(
    0.5, -0.05,
    "\n".join(textwrap.wrap(
        "CAVEAT: these patient-level numbers are NOT cleaner evidence than the slice-level ones above -- "
        "dataset perfectly predicts class in this project (all MSD patients cancer, all NIH healthy), so "
        "they inherit the same scanner-level confound at a coarser grain, and mean-aggregation itself is "
        "partly a symptom of it (near-flat within-patient scores -- see src/fusion/fusion.ipynb). "
        "KNOWN LIMITATION: A rigorous confound-check investigation found this model's tumour region "
        "moves its prediction significantly LESS than a same-sized random patch of ordinary tissue -- "
        "still open and unresolved. See checkpoints/imaging/final/model_card.json.", 140
    )),
    ha="center", va="top", fontsize=7, family="monospace",
)

pl_fig_path = EVAL_IMAGING_DIR / "patient_level_curves.png"
pl_fig.tight_layout()
pl_fig.savefig(pl_fig_path, dpi=100, bbox_inches="tight")
plt.close(pl_fig)
print(f"Saved {pl_fig_path}")
print(f"n_patients={len(pl_y_true)}  AUC={roc_auc_score(pl_y_true, pl_y_score):.4f}")

In [7]:
# Step 3: small in-sample sanity check (NOT a performance estimate -- see markdown above)
SANITY_SAMPLE_SIZE_PER_DATASET = 32

manifest_df = pd.read_csv(CACHE_MANIFEST_PATH)
with open(CACHE_META_PATH) as f:
    cache_meta = json.load(f)
box_size = cache_meta["box_size"]

sanity_sample = pd.concat([
    manifest_df[manifest_df["dataset"] == "MSD"].sample(SANITY_SAMPLE_SIZE_PER_DATASET, random_state=42),
    manifest_df[manifest_df["dataset"] == "NIH"].sample(SANITY_SAMPLE_SIZE_PER_DATASET, random_state=42),
]).reset_index(drop=True)

sanity_ds = SliceCacheDataset(sanity_sample, CACHE_IMAGES_PATH, CACHE_MASKS_PATH,
                               use_3channel=True, box_size=box_size, augment=False)
sanity_loader = DataLoader(sanity_ds, batch_size=16, shuffle=False, num_workers=0)

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in sanity_loader:
        images = batch["image"].to(DEVICE)
        _, det_logit = final_model(images)
        all_probs.append(torch.sigmoid(det_logit).cpu().numpy())
        all_labels.append(batch["label"].numpy())
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

sanity_results = {
    "n_sampled": int(len(all_probs)),
    "mean_p_cancer_given_true_cancer": float(all_probs[all_labels == 1].mean()),
    "mean_p_cancer_given_true_healthy": float(all_probs[all_labels == 0].mean()),
    "all_finite": bool(np.isfinite(all_probs).all()),
}

print("=== IN-SAMPLE SANITY CHECK -- NOT a performance estimate ===")
print("(this final model was fit on ALL data; there is no held-out set to score it on --")
print(" this only confirms the checkpoint produces sane, non-degenerate output)")
print(json.dumps(sanity_results, indent=2))

=== IN-SAMPLE SANITY CHECK -- NOT a performance estimate ===
(this final model was fit on ALL data; there is no held-out set to score it on --
 this only confirms the checkpoint produces sane, non-degenerate output)
{
  "n_sampled": 64,
  "mean_p_cancer_given_true_cancer": 0.9963425993919373,
  "mean_p_cancer_given_true_healthy": 0.01590249314904213,
  "all_finite": true
}


In [8]:
# Step 4: assemble and save model_card.json
comparison_metrics = {
    candidate: {
        "recall_mean": float(model_comparison_df.set_index("model").loc[candidate, "recall_mean"]),
        "roc_auc_mean": float(model_comparison_df.set_index("model").loc[candidate, "roc_auc_mean"]),
        "dice_mean": (
            None if np.isnan(model_comparison_df.set_index("model").loc[candidate, "dice_mean"])
            else float(model_comparison_df.set_index("model").loc[candidate, "dice_mean"])
        ),
    }
    for candidate in CANDIDATES
}

model_card = {
    "candidate": WINNER_MODEL_NAME,
    "architecture": "ResNet50UNet (pretrained ResNet-50 encoder + U-Net decoder + detection head)",
    "n_params": pod_run["n_params"],
    "fit_type": pod_run["fit_type"],
    "trained_on": date.today().isoformat(),
    "n_patients": pod_run["n_patients"],
    "n_rows": pod_run["n_rows"],
    "training_hyperparameters": {
        "n_epochs": pod_run["n_epochs"],
        "epoch_count_reasoning": pod_run["epoch_count_reasoning"],
        "batch_size": pod_run["batch_size"],
        "augment": pod_run["augment"],
        "slice_stride": pod_run["slice_stride"],
        "learning_rate": pod_run["learning_rate"],
        "optimizer": pod_run["optimizer"],
        "use_torch_compile": pod_run["use_torch_compile"],
        "box_size": pod_run["box_size"],
    },
    "comparison_metrics_that_justified_this_model": comparison_metrics,
    "selection_reasoning": (
        "resnet50_unet has substantially higher and more fold-stable recall (the clinically "
        "prioritised metric, since a missed cancer slice is costlier than a false alarm) than "
        "baseline_cnn, and is the only candidate producing segmentation output that the "
        "project's derived risk-score (tumour/gland pixel ratio) structurally depends on. "
        "See docs/Imaging_5Fold_Training_Results_documentation.md for the full comparison."
    ),
    "calibrator": "Platt scaling (LogisticRegression on raw sigmoid probability)",
    "calibrator_fit_rows": int(len(winner_oof)),
    "calibrator_fit_unique_patients": int(winner_oof["patient_id"].nunique()),
    "calibrator_reasoning": (
        "Platt scaling chosen over isotonic regression: isotonic typically needs thousands of "
        "samples per class to avoid staircase overfitting, and while there are 90,693 OOF rows, "
        "the effective unit of novel information is closer to the 361 unique patients, favouring "
        "the simpler, more stable parametric option. Fit on all slice-level OOF rows directly, "
        "never on the final model's own in-sample predictions, which would be circular."
    ),
    "in_sample_sanity_check": {
        "note": (
            "NOT a performance estimate -- this final model was fit on ALL 361 patients with no "
            "held-out data, so there is no true out-of-sample set to score it on. This is a small "
            "random subset checked only for sane, non-degenerate output. Real performance is only "
            "knowable via the 5-fold CV run backing this architecture/hyperparameters "
            "(checkpoints/imaging/candidates/resnet50_unet/model_card.json, summarized in "
            "comparison_metrics_that_justified_this_model above)."
        ),
        **sanity_results,
    },
    "known_limitations": {
        "confound_check_summary": (
            "A rigorous, 6-round confound-check investigation (Grad-CAM + 5 other attribution "
            "methods, then causal occlusion testing) found this architecture's detection head "
            "originally relied MORE on the BOX=320 packing's synthetic padding than on the real "
            "tumour region (Round 4: 2.03x a random-patch control, p=0.044). A random-resized-crop "
            "augmentation (augment=True, used in this final fit) verified-fixed the padding "
            "shortcut (Round 5: 1.08x control, p=0.917 -- statistically indistinguishable from the "
            "control). A second remediation attempt (Round 6: mask-preserving random erase) "
            "targeting the model's remaining UNDER-reliance on the tumour region itself did NOT "
            "help (0.58x control, p=0.0148, statistically unchanged from Round 5's 0.62x) and was "
            "reverted -- this promoted model does not include that change. "
            "TUMOUR UNDER-RELIANCE REMAINS OPEN AND UNRESOLVED in this promoted model: the real "
            "tumour region moves this model's prediction significantly LESS than a same-sized "
            "random patch of ordinary tissue. Detection metrics in this model card are strong and "
            "reproducible, but are NOT yet confirmed evidence of tumour-specific pixel-level "
            "reasoning. See docs/Imaging_Confound_Check_documentation.md (Rounds 1-6) and "
            "docs/Imaging_Session_Summary_2026-07-18.md for the complete investigation."
        ),
        "segmentation_variability": (
            "Segmentation Dice/IoU is weak and highly fold-variable in the backing CV run "
            "(fold Dice range ~0.06-0.54, mean 0.396) -- any downstream use of segmentation "
            "output (e.g. the derived risk-score) should treat it as a noisy signal, not a "
            "reliable per-slice measurement."
        ),
    },
}

with open(CHECKPOINTS_IMAGING_FINAL_DIR / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)

print(f"Saved model_card.json to {CHECKPOINTS_IMAGING_FINAL_DIR}")

Saved model_card.json to C:\FYP\checkpoints\imaging\final


In [9]:
# Step 5: confirm every promoted file is present
expected_files = ["model.pt", "calibrator.pkl", "model_card.json", "pod_training_run.json"]
all_present = all((CHECKPOINTS_IMAGING_FINAL_DIR / f).exists() for f in expected_files)
print(f"All {len(expected_files)} files present: {all_present}")
for f in expected_files:
    fpath = CHECKPOINTS_IMAGING_FINAL_DIR / f
    status = f"OK ({fpath.stat().st_size:,} bytes)" if fpath.exists() else "MISSING"
    print(f"  {status:<28} {f}")

All 4 files present: True
  OK (287,854,371 bytes)       model.pt
  OK (831 bytes)               calibrator.pkl
  OK (4,579 bytes)             model_card.json
  OK (3,413 bytes)             pod_training_run.json


## Confound Check — Summary (already investigated in full elsewhere)

In [10]:
occ_df = pd.read_csv(RESULTS_IMAGING_DIR / "confound_check_occlusion_sensitivity.csv")

mean_abs_tumor = occ_df["delta_tumor_logit"].abs().mean()
mean_abs_control = occ_df["delta_control_logit"].abs().mean()
tumor_vs_control = mean_abs_tumor / mean_abs_control

padding_qualifying = occ_df[occ_df["has_meaningful_padding"]]
mean_abs_padding = padding_qualifying["delta_padding_logit"].abs().mean()
mean_abs_control_padding_subset = padding_qualifying["delta_control_logit"].abs().mean()
padding_vs_control = mean_abs_padding / mean_abs_control_padding_subset

print(f"=== Occlusion sensitivity, recomputed from the live CSV (n={len(occ_df)} MSD slices) ===")
print(f"Mean |delta tumor logit|:                {mean_abs_tumor:.4f}")
print(f"Mean |delta control (random patch) logit|: {mean_abs_control:.4f}")
print(f"Tumor / control ratio: {tumor_vs_control:.2f}x "
      f"({'below 1x -- tumour moves the prediction LESS than a random patch' if tumor_vs_control < 1 else 'at/above 1x'})")
print()
print(f"Padding-qualifying slices (has_meaningful_padding): n={len(padding_qualifying)}")
print(f"Mean |delta padding logit|: {mean_abs_padding:.4f}")
print(f"Padding / control ratio: {padding_vs_control:.2f}x")
print()
print("(Significance testing -- paired Wilcoxon -- was already done in imaging_confound_check.ipynb;")
print(" this cell recomputes only the descriptive ratios directly from the same result file, as a live check")
print(" that this summary hasn't drifted from the actual data. p-values: see docs/Imaging_Confound_Check_documentation.md.)")

print("\n=== Verdict ===")
print("- Padding shortcut (model relied MORE on synthetic BOX=320 padding than the real tumour):")
print("    FIXED & VERIFIED (Round 5) -- 2.03x control, p=0.044 -> 1.08x, p=0.917.")
print("- Tumour under-reliance (model relies LESS on the real tumour than a random patch of tissue):")
print("    STILL OPEN. Two remediation attempts made (crop-augment: fixed padding but not this;")
print("    mask-preserving erase, Round 6: did not help, reverted). Not a new finding -- disclosed")
print("    explicitly in checkpoints/imaging/final/model_card.json's known_limitations.")
print("\nFull detail: docs/Imaging_Confound_Check_documentation.md, docs/Imaging_Session_Summary_2026-07-18.md")

=== Occlusion sensitivity, recomputed from the live CSV (n=65 MSD slices) ===
Mean |delta tumor logit|:                0.2119
Mean |delta control (random patch) logit|: 0.3416
Tumor / control ratio: 0.62x (below 1x -- tumour moves the prediction LESS than a random patch)

Padding-qualifying slices (has_meaningful_padding): n=58
Mean |delta padding logit|: 0.3701
Padding / control ratio: 1.08x

(Significance testing -- paired Wilcoxon -- was already done in imaging_confound_check.ipynb;
 this cell recomputes only the descriptive ratios directly from the same result file, as a live check
 that this summary hasn't drifted from the actual data. p-values: see docs/Imaging_Confound_Check_documentation.md.)

=== Verdict ===
- Padding shortcut (model relied MORE on synthetic BOX=320 padding than the real tumour):
    FIXED & VERIFIED (Round 5) -- 2.03x control, p=0.044 -> 1.08x, p=0.917.
- Tumour under-reliance (model relies LESS on the real tumour than a random patch of tissue):
    STILL OPE